# ✅ Solusi — Tokenizer & Mini-LM dari Nol (Bab 9)

> Buka HANYA setelah starter selesai dikerjakan / benar-benar mentok.
> Struktur sama dengan starter, plus **jawaban analisis** di tiap eksperimen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tok_mini.bpe import latih_bpe, normalisasi
from tok_mini.data import CORPUS_TEKS, LM_D, LM_SEQ, LM_VOCAB, SAMPLE_LOGITS, SEED
from tok_mini.embedding import analogi, cari_mirip, cos_sim, latih_embedding
from tok_mini.lm import init_params, prediksi_berikut, train_lm
from tok_mini.attention import self_attention
from tok_mini.sampling import sample_greedy, sample_topp, sample_topk
from tok_mini.tokenizer import buat_vocab, encode_teks

CORPUS = normalisasi(CORPUS_TEKS)
merges, _ = latih_bpe(CORPUS, 4)
vocab = buat_vocab(CORPUS, 4)
print('siap. SEED =', SEED)

## Eksperimen 1 — Anatomi Token

In [ ]:
teks = "newer lowliest newer newest wider low lowlier"
hasil = encode_teks(teks, merges, vocab)
for kata, toks in zip(normalisasi(teks), hasil):
    print(f'{kata:<9} ({len(toks)} token) -> {list(toks)}')
total_token = sum(len(t) for t in hasil)
print(f'\nratio token/kata: {total_token / len(hasil):.2f}')

**Jawaban 1.** Paling hemat: `newer` (2 token — `new` + `er</w>`) karena muncul 2x di
corpus sehingga BPE menemukan `n+ew` lalu `new`. Paling boros: `lowliest` (9 token —
praktis per karakter) karena tidak pernah muncul di corpus. `lowlier` juga berat:
kata baru meski mirip `lower`. Tetap TIDAK ada error — semua potongan ada di vocab
(fallback karakter); inilah yang membuat API LLM bisa memproses teks apa pun, dan
kenapa biaya dihitung per token: kata yang sama bisa 2x atau 9x lebih mahal tergantung
seberapa 'umum' ia di data training tokenizer.

## Eksperimen 2 — Trade-off Jumlah Merge

In [ ]:
uji = "newer lowliest newer newest wider"
print(f'{"n_merge":>7} | {"vocab":>5} | {"total token":>11}')
print('-' * 34)
for n in (0, 1, 2, 4, 8):
    m, _ = latih_bpe(CORPUS, n)
    v = buat_vocab(CORPUS, n)
    tot = sum(len(t) for t in encode_teks(uji, m, v))
    print(f'{n:>7} | {len(v):>5} | {tot:>11}')

**Jawaban 2.** Total token teks uji: 34 -> 31 -> 28 -> 22 -> 19 untuk
n = 0, 1, 2, 4, 8. Turun cepat di awal, melandai di akhir. Merge pertama (`er`)
menghemat 3 token sendirian — pasangan terfrekuensi memberi penghematan terbesar;
merge berikutnya makin spesifik. Tokenizer produksi berhenti di ~50rb–250rb karena:
(1) diminishing returns — merge ke-100rb jarang dipakai, (2) tabel embedding
membesar dengan vocab (parameter & memori), (3) vocab besar merusak token bahasa
minoritas yang jarang ada di data. Trade-off yang sama persis dengan eksperimen ini.

## Eksperimen 3 — Struktur di Ruang Embedding

In [ ]:
W = latih_embedding(idx_vokal=(0, 4, 6), vocab=8, d=8, epochs=200, seed=SEED)
C = np.array([[cos_sim(W[i], W[j]) for j in range(8)] for i in range(8)])
print(np.round(C, 2))
print('\ncari_mirip(token 0):', cari_mirip(W, W[0], exclude={0}, top_n=3))
print('analogi(1, 0, 2):', analogi(1, 0, 2, W))

W64 = latih_embedding(idx_vokal=(0, 4, 6), vocab=8, d=64, epochs=200, seed=SEED)
d64_dalem = np.mean([cos_sim(W64[i], W64[j]) for i in (0, 4, 6) for j in (0, 4, 6) if i < j])
d64_luar = np.mean([cos_sim(W64[i], W64[j]) for i in (0, 4, 6) for j in (1, 2, 5)])
print(f'\nd=8 : dalam {C[0,4]:.3f} | d=64: dalam {d64_dalem:.3f}, luar {d64_luar:.3f}')

**Jawaban 3.** Blok terang di persilangan baris/kolom {0, 4, 6} — kelompok yang
menerima signal yang sama selama 'training'. Tidak ada label: struktur muncul karena
DATA menggeser vektor ke arah yang sama — persis mekanisme word2vec. Eksperimen
dimensi: vektor acak 60 pasang punya std cosine 0.356 di d=8 vs 0.126 di d=64 —
di dimensi tinggi vektor acak hampir ORTOGONAL (bandingkan dengan muat-muatan noise
di d=8 yang lebar). Struktur kelompok (dalam=1.00 vs luar 0.14 di d=64) masih
terbaca JUSTRU karena kontrasnya tinggi terhadap noise yang makin sempit. Implikasi
RAG (Bab 12): di ruang berdimensi besar, 'mirip' lebih mudah dibedakan dari noise —
tapi model tetap butuh BANYAK contoh per konsep agar cluster terbentuk.

## Eksperimen 4 — Attention: Encoder vs Decoder

In [ ]:
rng = np.random.default_rng(SEED)
Q = rng.normal(0, 1, (6, 8)); K = rng.normal(0, 1, (6, 8)); V = rng.normal(0, 1, (6, 8))
out_enc, attn_enc = self_attention(Q, K, V, causal=False)
out_dec, attn_dec = self_attention(Q, K, V, causal=True)
print('baris terakhir identik?', np.allclose(attn_enc[-1], attn_dec[-1]))
print('baris pertama identik? ', np.allclose(attn_enc[0], attn_dec[0]))
print('baris ke-3 identik?    ', np.allclose(attn_enc[3], attn_dec[3]))
print('attn_dec baris 0:', attn_dec[0].round(3))

**Jawaban 4.** Hanya baris TERAKHIR yang identik: ia tidak punya masa depan, jadi mask
tidak mengubah apa pun. Baris awal paling parah — hamper semua kolomnya terlarang
(baris 0 = [1,0,0,0,...]). Kalau mask dilupakan saat training GPT tapi dipakai saat
inference: training melihat token t+1 (jawaban bocor), inference tidak — distribusi
input training dan inference BEDA, generasi keluar dari distribusi yang pernah
dipelajari → teks hancur cepat, walau loss training tampak sangat bagus.

## Eksperimen 5 — Distribusi Sampling

In [ ]:
logits = np.asarray(SAMPLE_LOGITS)
rng = np.random.default_rng(SEED)
strategi = {
    'greedy': [sample_greedy(logits)] * 300,
    'top-k=4': [sample_topk(logits, 4, rng) for _ in range(300)],
    'top-p=0.6': [sample_topp(logits, 0.6, rng) for _ in range(300)],
}
for nama, sampel in strategi.items():
    unik = sorted(set(sampel))
    print(f'{nama:<10}: token unik = {unik}')

rng2 = np.random.default_rng(0)
for T in (0.5, 1.0, 2.0):
    ids = [sample_topk(logits, 4, rng2, T=T) for _ in range(1000)]
    p1 = ids.count(1) / len(ids)
    print(f'T={T:>4}: proporsi token 1 = {p1:.2f}')

**Jawaban 5.** Greedy = satu batang (selalu token 1). top-p=0.6 memotong nucleus jadi
{1, 3, 5} — tiga batang, sesuai bobot softmax lokalnya. Untuk chatbot layanan
pelanggan: T rendah (~0.2–0.5) + top-p ~0.9 — jawaban harus konsisten dan patuh
prosedur; risiko T rendah: repetitif & kaku untuk pertanyaan aneh, dan tetap butuh
guardrail karena bukan berarti selalu benar.

## Eksperimen 6 — Generalisasi Mini-LM

In [ ]:
trained, losses = train_lm(init_params(SEED, vocab=LM_VOCAB, d=LM_D), LM_SEQ,
                           epochs=120, lr=0.05)
print(f'loss: {losses[0]:.3f} -> {losses[-1]:.4f} (baseline buta ln(4) = 1.386)')
for pola in ([0], [0, 1], [1], [1, 0], [0, 1, 0, 1]):
    print(f'{pola} -> prediksi {prediksi_berikut(trained, pola)}')

**Jawaban 6.** `[1] -> 0`, `[1, 0] -> 1`, bahkan `[0, 1, 0, 1] -> 0` — semua benar
meski training cuma pernah melihat 8-token yang mulai dari 0. Model tidak menghafal
posisi ('posisi ke-1 = token 0') melainkan hubungan BERGANTIAN (token sekarang ≠
token sebelumnya) lewat embedding We yang berbeda per token. Itulah mengapa LLM
besar bisa melanjutkan teks yang tak pernah ada di data training: yang dipelajari
adalah pola statistik hubungan antar token (bahasa, logika, format), bukan indeks
hafalan. (Generalisasi mini-LM ini sempit — hanya pola bergantian; LLM besar
memperluasnya dengan arsitektur yang sama + skala data raksasa.)

## Ringkasan

| Eksperimen | Temuan kunci | Angka |
|---|---|---|
| 1 | kata sering hemat, kata baru pecah per karakter | `newer` 2 token vs `lowliest` 9 |
| 2 | kurva merge: turun cepat lalu melandai | n=0: 34 token -> n=8: 19 token |
| 3 | struktur muncul dari data, bukan label | cos dalam 1.00 vs luar 0.07 (d=8) |
| 4 | hanya baris terakhir encoder=decoder | baris 0 decoder = [1,0,0,...] |
| 5 | greedy 1 token; top-p=0.6 -> nucleus {1,3,5} | T=2.0: proporsi token 1 = 0.39 |
| 6 | model pelajari RELASI, bukan posisi | `[1] -> 0` benar |

Selesai? Isi `RUBRIK.md` — lalu lanjut ke **Bab 10 (Prompt Engineering)**.